In [1]:
%pip install -quiet PyPDF2 opensearch-py sentence-transformers

Note: you may need to restart the kernel to use updated packages.



Usage:   
  c:\Users\ksuma\OneDrive\Documents\local-rag-system\.venv\Scripts\python.exe 
-m pip install [options] <requirement specifier> [package-index-options] ...
  c:\Users\ksuma\OneDrive\Documents\local-rag-system\.venv\Scripts\python.exe 
-m pip install [options] -r <requirements file> [package-index-options] ...
  c:\Users\ksuma\OneDrive\Documents\local-rag-system\.venv\Scripts\python.exe 
-m pip install [options] [-e] <vcs project url> ...
  c:\Users\ksuma\OneDrive\Documents\local-rag-system\.venv\Scripts\python.exe 
-m pip install [options] [-e] <local project path> ...
  c:\Users\ksuma\OneDrive\Documents\local-rag-system\.venv\Scripts\python.exe 
-m pip install [options] <archive url/path> ...

no such option: -u


In [2]:
import json
import re
import os
import sys
from typing import Dict, List, Any

from PyPDF2 import PdfReader
from opensearchpy import OpenSearch, helpers
from sentence_transformers import SentenceTransformer

c:\Users\ksuma\OneDrive\Documents\local-rag-system\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
OPENSEARCH_HOST = "localhost"
OPENSEARCH_PORT = 9200
OPENSEARCH_INDEX ="documents"

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
EMBEDDING_DIMENSIONS = 384
ASYMETRIC_EMBEDDING = False

TEXT_CHUNK_SIZE = 500
TEXT_CHUNK_OVERLAP = 100


In [4]:
def clean_text(text: str) -> str:
    """Cleans OCR-extracted text by removing unnecessary newlines, hyphens, and correcting common OCR errors.

    Args:
        text (str): The text to clean.

    Returns:
        str: The cleaned text."""

    text = re.sub(r"(\w+)-\n(\w+)", r"\1\2", text)
    text = re.sub(r"(?<!\n)\n(?!\n)", " ", text)
    text = re.sub(r"\n+", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)

    return text.strip()

In [5]:
def chunk_text(text: str, chunk_size: int, overlap: int = 100) -> List[str]:
    """Splits text into chunks with a specified overlap.

    Args:
        text (str): The text to split.
        chunk_size (int): The number of tokens in each chunk.
        overlap (int): The number of tokens to overlap between chunks.

    Returns:
        List[str]: A list of text chunks."""

    text = clean_text(text)
    tokens = text.split(" ")
    chunks = []
    start = 0
    while start <= len(tokens):
        end = start + chunk_size
        chunk_token = tokens[start:end]
        chunk_text = " ".join(chunk_token)
        chunks.append(chunk_text)
        start = end - overlap
    
    return chunks

In [6]:
def get_embedding_model():
    """Loads and returns the sentence transformer embedding model.
    
    Returns:
        SentenceTransformer: The loaded embedding model."""
    model = SentenceTransformer(EMBEDDING_MODEL)
    return model

def generate_embedding(texts: List[str]):
    """
    Generates embeddings for a list of text chunks.
    
    Args:
        texts (List[str]): List of text chunks to embed.
        
    Returns:
        List[numpy.ndarray]: List of embedding vectors.
    """
    model = get_embedding_model()

    if ASYMETRIC_EMBEDDING :
        texts = [f"passage : {text}" for text in texts]

    embeddings = model.encode(
    texts,
    convert_to_numpy=True,
    show_progress_bar=True,
    )
    return embeddings
    

In [7]:
client = OpenSearch(
    hosts = [{"hosts": OPENSEARCH_HOST, "port": OPENSEARCH_PORT}],
    http_compress = True,
    timeout = 30,
    max_retries = 3,
    retry_on_timeout = True,
)
try:
    info = client.info()
    print(f"Successfully connect to OpenSearch {info['version']['number']}")
except Exception as e:
    print(f"Failed to connect to OpenSearch: {e}")
    print("Make sure OpenSearch is running on localhost:9200")
    raise


Successfully connect to OpenSearch 2.19.2


In [8]:
def create_index_config() -> Dict[str, Any]:
    """
    Creates the index configuration with mappings for text, embeddings, and metadata.
    
    Returns:
        Dict[str, Any]: The index configuration.
    """
    config = {
        "settings": {
            "index": {
                "number_of_shards": 1,
                "number_of_replicas": 0,
                "knn": True,
            }
        },
        "mappings":{
            "properties": {
                "text": {
                    "type": "text"
                },
                "embedding": {
                    "type": "knn_vector",
                    "dimension": EMBEDDING_DIMENSIONS,
                    "method": {
                        "engine":"faiss",
                        "space_type": "l2",
                        "name": "hnsw",
                        "parameters": {}
                    }
                },
                "document_name": {
                    "type": "keyword"
                }
            }
        }
    }
    return config

index_config = create_index_config()
print("\nIndex Configuration:")
print(json.dumps(index_config, indent=2))


Index Configuration:
{
  "settings": {
    "index": {
      "number_of_shards": 1,
      "number_of_replicas": 0,
      "knn": true
    }
  },
  "mappings": {
    "properties": {
      "text": {
        "type": "text"
      },
      "embedding": {
        "type": "knn_vector",
        "dimension": 384,
        "method": {
          "engine": "faiss",
          "space_type": "l2",
          "name": "hnsw",
          "parameters": {}
        }
      },
      "document_name": {
        "type": "keyword"
      }
    }
  }
}


In [9]:
if not client.indices.exists(index=OPENSEARCH_INDEX):
    response = client.indices.create(index=OPENSEARCH_INDEX, body=index_config)
    print(f"\nCreated index {OPENSEARCH_INDEX} with response: {response}")
else:
    print(f"\nIndex {OPENSEARCH_INDEX} already exists")


Index documents already exists


In [10]:
from opensearchpy.exceptions import NotFoundError
pipeline_name = "nlp-search-pipeline"

try:
    result = client.transport.perform_request(
        "GET",
        f"/_search/pipeline/{pipeline_name}"
    )
    print(f"\n Search pipeline {pipeline_name} exists.")
    print(result)
except NotFoundError:
    print(f"\n Search pipeline {pipeline_name} not exists.")
except Exception as e:
    print(f"\n error {e}")


 Search pipeline nlp-search-pipeline exists.


{'nlp-search-pipeline': {'description': 'Post processor for hybrid search', 'phase_results_processors': [{'normalization-processor': {'normalization': {'technique': 'min_max'}, 'combination': {'technique': 'arithmetic_mean', 'parameters': {'weights': [0.3, 0.7]}}}}]}}


In [11]:
pdf_path = "attention is all you need.pdf"

reader = PdfReader(pdf_path)
text = "".join(page.extract_text() for page in reader.pages)
print(f"Extracted {len(text)} characters from {pdf_path}")

print("\nSample of extracted text:")
print(text[:500] + "...")

cleaned_text = clean_text(text)
print(f"\nlen of cleaned text {len(cleaned_text)}")

chunks = chunk_text(cleaned_text, chunk_size=TEXT_CHUNK_SIZE, overlap=TEXT_CHUNK_OVERLAP)
print(f"Split text into {len(chunks)} chunks")

print("\nSample chunk:")
print(chunks[0])

Extracted 39472 characters from attention is all you need.pdf

Sample of extracted text:
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.comNoam Shazeer∗
Google Brain
noam@google.comNiki Parmar∗
Google Research
nikip@google.comJakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.comAidan N. Gomez∗ †
University of Toronto
aidan@cs.toronto.eduŁukasz Kaise...

len of cleaned text 39437
Split text into 15 chunks

Sample chunk:
Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works. Attention Is All You Need Ashish Vaswani∗ Google Brain avaswani@google.comNoam Shazeer∗ Google Brain noam@google.comNiki Parmar∗ Google Research nikip@google.comJako

In [12]:
pdf_file_name = pdf_path.replace(".pdf","")

embeddings = generate_embedding(chunks)
print(f"Generated {len(embeddings)} embeddings")
print(f"Embedding shape: {embeddings[0].shape}")
print(embeddings[0])

documents_to_index =[{
    "doc_id": f"{pdf_file_name}_{i}",
    "text": chunk,
    "embedding": embedding,
    "document_name": pdf_file_name,
    } for i, (chunk, embedding) in enumerate(zip(chunks, embeddings))
]
print(f"\nPrepared {len(documents_to_index)} documents for indexing")

Batches: 100%|██████████| 1/1 [00:01<00:00,  1.73s/it]

Generated 15 embeddings
Embedding shape: (384,)
[-1.09950088e-01 -1.12957157e-01  3.23199593e-02 -2.13630386e-02
  4.98477407e-02  4.77034412e-02 -6.99672848e-02  5.11832163e-03
  1.35308683e-01 -2.16666013e-02  1.35875782e-02 -4.90982980e-02
 -5.04511781e-02  4.11290601e-02 -8.15893337e-02  4.39033797e-03
  1.59146413e-02  4.93085608e-02 -6.03301451e-02 -7.90917352e-02
  2.21454278e-02  6.37505502e-02  1.53672984e-02  5.12807397e-03
 -2.96906033e-03 -6.83625741e-03 -6.33631647e-02 -4.30668741e-02
 -1.55117991e-03  2.68965121e-02  2.08034739e-02 -2.84459330e-02
  7.60252951e-05  9.52623039e-02 -5.79076670e-02  6.08346723e-02
 -6.33520484e-02 -1.19274333e-02  2.96378545e-02 -3.67110968e-02
  2.83527132e-02 -2.68741418e-02 -1.39877712e-02  3.06291971e-02
  1.49389207e-01  4.69272174e-02 -3.26575572e-03 -1.05649596e-02
 -1.27882306e-02  1.02554057e-02 -9.48563740e-02  8.42149556e-02
  1.75420549e-02  1.10733218e-01 -4.72708140e-03 -1.28204767e-02
 -1.76323354e-02 -2.55317409e-02 -4.111218

In [13]:
documents_to_index

[{'doc_id': 'attention is all you need_0',
  'text': 'Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works. Attention Is All You Need Ashish Vaswani∗ Google Brain avaswani@google.comNoam Shazeer∗ Google Brain noam@google.comNiki Parmar∗ Google Research nikip@google.comJakob Uszkoreit∗ Google Research usz@google.com Llion Jones∗ Google Research llion@google.comAidan N. Gomez∗ † University of Toronto aidan@cs.toronto.eduŁukasz Kaiser∗ Google Brain lukaszkaiser@google.com Illia Polosukhin∗ ‡ illia.polosukhin@gmail.com Abstract The dominant sequence transduction models are based on complex recurrent or convolutional neural networks that include an encoder and a decoder. The best performing models also connect the encoder and decoder through an attention mechanism. We propose a new simple network architecture, the Transformer, based solely on attention mechanisms, dispensi

In [15]:
actions = []

for doc in documents_to_index:
    if ASYMETRIC_EMBEDDING:
        prefixed_text = f"passage:{doc['text']}"
    else:
        prefixed_text = doc['text']
    action = {
        "_index": OPENSEARCH_INDEX,
        "_id": doc["doc_id"],
        "_source": {
            "text": prefixed_text,
            "embedding": doc["embedding"],
            "document_name": doc['document_name'],
        }
    }
    actions.append(action)

In [17]:
try:
    success, errors = helpers.bulk(client, actions, raise_on_error=True)
    if errors:
        print(f"Indexed {success} documents with {len(errors)} errors")
        print(f"First error: {errors[0]}")
    else:
        print(f"Successfully indexed {success} documents")
except Exception as e:
    print(f"Error during bulk loading {e}")

Successfully indexed 15 documents


In [18]:
query_text = "Model Architure"

query_embedding = generate_embedding([query_text])[0].tolist()

top_k = 3

query_body = {
    "_source": {"exclude": ["embedding"]},
    "query": {
        "hybrid": {
            "queries": [
                {"match": {"text": {"query": query_text}}},
                {
                    "knn": {
                        "embedding": {
                            "vector": query_embedding,
                            "k": top_k
                        }
                    }
                }
            ]
        }
    },
    "size": top_k
}

response = client.search(
        index=OPENSEARCH_INDEX, body=query_body, search_pipeline="nlp-search-pipeline"
    )

print(f"\nTop {top_k} results for query: '{query_text}'\n")
for i, hit in enumerate(response['hits']['hits'], 1):
    print(f"Result {i}:")
    print(json.dumps(hit['_source'], indent=2))
    print("-" * 60)

Batches: 100%|██████████| 1/1 [00:00<00:00,  6.98it/s]



Top 3 results for query: 'Model Architure'

Result 1:
{
  "document_name": "attention is all you need",
  "text": "models on the English-to-German and English-to-French newstest2014 tests at a fraction of the training cost. ModelBLEU Training Cost (FLOPs) EN-DE EN-FR EN-DE EN-FR ByteNet [18] 23.75 Deep-Att + PosUnk [39] 39.2 1.0\u00b71020 GNMT + RL [38] 24.6 39.92 2.3\u00b710191.4\u00b71020 ConvS2S [9] 25.16 40.46 9.6\u00b710181.5\u00b71020 MoE [32] 26.03 40.56 2.0\u00b710191.2\u00b71020 Deep-Att + PosUnk Ensemble [39] 40.4 8.0\u00b71020 GNMT + RL Ensemble [38] 26.30 41.16 1.8\u00b710201.1\u00b71021 ConvS2S Ensemble [9] 26.36 41.29 7.7\u00b710191.2\u00b71021 Transformer (base model) 27.3 38.1 3.3\u00b71018 Transformer (big) 28.4 41.8 2.3\u00b71019 Residual Dropout We apply dropout [ 33] to the output of each sub-layer, before it is added to the sub-layer input and normalized. In addition, we apply dropout to the sums of the embeddings and the positional encodings in both the encoder a